<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260316_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
apt install -y mariadb-server mariadb-client -q
service mariadb start

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  galera-4 gawk libcgi-fast-perl libcgi-pm-perl libclone-perl
  libconfig-inifiles-perl libdbd-mysql-perl libdbi-perl libencode-locale-perl
  libfcgi-bin libfcgi-perl libfcgi0ldbl libhtml-parser-perl
  libhtml-tagset-perl libhtml-template-perl libhttp-date-perl
  libhttp-message-perl libio-html-perl liblwp-mediatypes-perl libmariadb3
  libterm-readkey-perl liburi-perl liburing2 mariadb-client-10.6
  mariadb-client-core-10.6 mariadb-common mariadb-server-10.6
  mariadb-server-core-10.6
Suggested packages:
  gawk-doc libmldbm-perl libnet-daemon-perl libsql-statement-perl
  libdata-dump-perl libipc-sharedcache-perl libbusiness-isbn-perl libwww-perl
  mailx mariadb-test netcat-openbsd
The following NEW packages will be installed:
  galera-4 gawk libcgi-fast-perl libcgi-pm-perl libclone-perl
  libconfig-inifiles-perl libdbd-mysql-perl libdbi-perl libencode-l

In [2]:
%%bash
mysql -u root -e "SELECT 1;"

1
1


In [3]:
%%bash
mysql -u root -e "
CREATE DATABASE IF NOT EXISTS testdb;
CREATE USER IF NOT EXISTS 'testuser'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON testdb.* TO 'testuser'@'localhost';
FLUSH PRIVILEGES;"

mysql -u root testdb -e "
CREATE TABLE IF NOT EXISTS member (
    member_id INT PRIMARY KEY AUTO_INCREMENT,
    user_id VARCHAR(50) NOT NULL UNIQUE,
    user_pw VARCHAR(100) NOT NULL,
    user_name VARCHAR(50) NOT NULL,
    email VARCHAR(100),
    phone VARCHAR(20),
    join_date DATETIME DEFAULT CURRENT_TIMESTAMP
);"

In [4]:
%%bash
cd /content/project
npm init -y
npm install express mysql2

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}




added 76 packages, and audited 77 packages in 6s

24 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


bash: line 1: cd: /content/project: No such file or directory




---



In [5]:
import os
os.makedirs("/content/project/templates", exist_ok=True)

In [6]:
%%writefile /content/project/templates/signup.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>회원가입</title>
    <style>
        * { box-sizing: border-box; }
        body { margin: 0; font-family: Arial, sans-serif; background-color: #f4f6f8; }
        .container { width: 420px; margin: 60px auto; background: #ffffff; padding: 30px; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.08); }
        h1 { text-align: center; margin-bottom: 24px; }
        .form-group { margin-bottom: 16px; }
        label { display: block; margin-bottom: 8px; font-weight: bold; }
        input { width: 100%; padding: 12px; border: 1px solid #cccccc; border-radius: 8px; font-size: 14px; }
        button { width: 100%; padding: 14px; border: none; border-radius: 8px; background-color: #2563eb; color: white; font-size: 16px; cursor: pointer; }
        button:hover { background-color: #1d4ed8; }
    </style>
</head>
<body>
    <div class="container">
        <h1>회원가입</h1>
        <form action="/signup" method="post">
            <div class="form-group">
                <label for="user_id">아이디</label>
                <input type="text" id="user_id" name="user_id" required>
            </div>
            <div class="form-group">
                <label for="user_pw">비밀번호</label>
                <input type="password" id="user_pw" name="user_pw" required>
            </div>
            <div class="form-group">
                <label for="user_name">이름</label>
                <input type="text" id="user_name" name="user_name" required>
            </div>
            <div class="form-group">
                <label for="email">이메일</label>
                <input type="email" id="email" name="email">
            </div>
            <div class="form-group">
                <label for="phone">전화번호</label>
                <input type="text" id="phone" name="phone">
            </div>
            <button type="submit">회원가입</button>
        </form>
    </div>
</body>
</html>

Writing /content/project/templates/signup.html


In [7]:
%%writefile /content/project/templates/signup-success.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>회원가입 완료</title>
    <style>
        body { margin: 0; font-family: Arial, sans-serif; background-color: #f0fdf4; }
        .container { width: 420px; margin: 100px auto; background: #ffffff; padding: 40px; border-radius: 12px; text-align: center; box-shadow: 0 4px 12px rgba(0,0,0,0.08); }
        h1 { color: #15803d; }
        p { margin-top: 16px; font-size: 16px; }
        a { display: inline-block; margin-top: 24px; padding: 12px 20px; background-color: #2563eb; color: white; text-decoration: none; border-radius: 8px; }
    </style>
</head>
<body>
    <div class="container">
        <h1>회원가입 완료</h1>
        <p>회원 정보가 정상적으로 저장되었습니다.</p>
        <a href="/">다시 회원가입하기</a>
    </div>
</body>
</html>

Writing /content/project/templates/signup-success.html


In [8]:
%%writefile /content/project/server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2');

const app = express();
const PORT = 3000;

app.use(express.urlencoded({ extended: true }));

const conn = mysql.createConnection({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

conn.connect((err) => {
    if (err) {
        console.error('DB 연결 실패:', err);
        return;
    }
    console.log('DB 연결 성공');
});

// 회원가입 페이지
app.get('/', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'signup.html'));
});

// 회원가입 성공 페이지
app.get('/signup-success', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'signup-success.html'));
});

// 로그인 페이지
app.get('/login', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'login.html'));
});

// 로그인 성공 페이지
app.get('/login-success', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'login-success.html'));
});

// 로그인 실패 페이지
app.get('/login-fail', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'login-fail.html'));
});

// 회원가입 처리
app.post('/signup', (req, res) => {
    const { user_id, user_pw, user_name, email, phone } = req.body;

    const sql = `
        INSERT INTO member (user_id, user_pw, user_name, email, phone)
        VALUES (?, ?, ?, ?, ?)
    `;

    const values = [user_id, user_pw, user_name, email, phone];

    conn.query(sql, values, (err, result) => {
        if (err) {
            console.error('회원가입 INSERT 실패:', err);
            return res.send(`
                <h1>회원가입 실패</h1>
                <p>데이터 저장 중 오류가 발생했습니다.</p>
                <p>${err.message}</p>
                <a href="/">회원가입 페이지로 돌아가기</a>
            `);
        }

        console.log('회원가입 성공:', result);
        res.redirect('/login');
    });
});

// 로그인 처리
app.post('/login', (req, res) => {
    const { user_id, user_pw } = req.body;

    const sql = `
        SELECT * FROM member
        WHERE user_id = ?
    `;

    conn.query(sql, [user_id], (err, rows) => {
        if (err) {
            console.error('로그인 SELECT 실패:', err);
            return res.send(`
                <h1>로그인 실패</h1>
                <p>DB 조회 중 오류가 발생했습니다.</p>
                <p>${err.message}</p>
                <a href="/login">로그인 페이지로 돌아가기</a>
            `);
        }

        if (rows.length === 0) {
            return res.redirect('/login-fail');
        }

        const member = rows[0];

        if (member.user_pw === user_pw) {
            return res.redirect('/login-success');
        } else {
            return res.redirect('/login-fail');
        }
    });
});

app.listen(PORT, () => {
    console.log('=========================================');
    console.log(` 회원 서비스 서버가 포트 ${PORT}에서 작동 중입니다.`);
    console.log('=========================================');
});

Writing /content/project/server.js


In [9]:
%%bash
cd /content/project
npm init -y
npm install express mysql2

Wrote to /content/project/package.json:

{
  "name": "project",
  "version": "1.0.0",
  "main": "server.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1",
    "start": "node server.js"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}




added 76 packages, and audited 77 packages in 2s

24 packages are looking for funding
  run `npm fund` for details

found 0 vulnerabilities


In [10]:
%%bash
npm install -g cloudflared


added 1 package in 2s


In [11]:
%%bash
cd /content/project
nohup node server.js > server.log 2>&1 &
sleep 2
tail -n 20 server.log

 회원 서비스 서버가 포트 3000에서 작동 중입니다.
DB 연결 성공


In [12]:
%%bash
nohup cloudflared tunnel --url http://localhost:3000 > /content/project/tunnel.log 2>&1 &

In [17]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" /content/project/tunnel.log | tail -n 1

https://drums-improvement-gone-wherever.trycloudflare.com




---



In [14]:
%%writefile /content/project/templates/login.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>로그인</title>
    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background-color: #eef2f7;
        }

        .container {
            width: 400px;
            margin: 100px auto;
            background: #ffffff;
            padding: 30px;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
        }

        h1 {
            text-align: center;
            margin-bottom: 24px;
        }

        .form-group {
            margin-bottom: 16px;
        }

        label {
            display: block;
            margin-bottom: 8px;
            font-weight: bold;
        }

        input {
            width: 100%;
            padding: 12px;
            border: 1px solid #cccccc;
            border-radius: 8px;
            font-size: 14px;
        }

        button {
            width: 100%;
            padding: 14px;
            border: none;
            border-radius: 8px;
            background-color: #16a34a;
            color: white;
            font-size: 16px;
            cursor: pointer;
        }

        button:hover {
            background-color: #15803d;
        }

        .link-box {
            text-align: center;
            margin-top: 16px;
        }

        .link-box a {
            text-decoration: none;
            color: #2563eb;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>로그인</h1>

        <form action="/login" method="post">
            <div class="form-group">
                <label for="user_id">아이디</label>
                <input type="text" id="user_id" name="user_id" required>
            </div>

            <div class="form-group">
                <label for="user_pw">비밀번호</label>
                <input type="password" id="user_pw" name="user_pw" required>
            </div>

            <button type="submit">로그인</button>
        </form>

        <div class="link-box">
            <a href="/">회원가입 페이지로 이동</a>
        </div>
    </div>
</body>
</html>

Writing /content/project/templates/login.html


In [15]:
%%writefile /content/project/templates/login-success.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>로그인 성공</title>
    <style>
        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background-color: #f0fdf4;
        }

        .container {
            width: 420px;
            margin: 100px auto;
            background: #ffffff;
            padding: 40px;
            border-radius: 12px;
            text-align: center;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
        }

        h1 {
            color: #15803d;
        }

        p {
            margin-top: 16px;
            font-size: 16px;
        }

        a {
            display: inline-block;
            margin-top: 24px;
            padding: 12px 20px;
            background-color: #2563eb;
            color: white;
            text-decoration: none;
            border-radius: 8px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>로그인 성공</h1>
        <p>회원 인증이 정상적으로 완료되었습니다.</p>
        <a href="/login">다시 로그인하기</a>
    </div>
</body>
</html>

Writing /content/project/templates/login-success.html


In [16]:
%%writefile /content/project/templates/login-fail.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>로그인 실패</title>
    <style>
        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background-color: #fef2f2;
        }

        .container {
            width: 420px;
            margin: 100px auto;
            background: #ffffff;
            padding: 40px;
            border-radius: 12px;
            text-align: center;
            box-shadow: 0 4px 12px rgba(0,0,0,0.08);
        }

        h1 {
            color: #dc2626;
        }

        p {
            margin-top: 16px;
            font-size: 16px;
        }

        a {
            display: inline-block;
            margin-top: 24px;
            padding: 12px 20px;
            background-color: #2563eb;
            color: white;
            text-decoration: none;
            border-radius: 8px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>로그인 실패</h1>
        <p>아이디 또는 비밀번호가 올바르지 않습니다.</p>
        <a href="/login">로그인 페이지로 돌아가기</a>
    </div>
</body>
</html>

Writing /content/project/templates/login-fail.html
